# Palindrome Sentences with Ollama

This notebook uses [Ollama](https://ollama.com) — a local LLM server — to explore three approaches
to generating palindrome sentences:

1. **Direct prompting** — ask the model politely and see what it produces
2. **Iterative refinement** — Python checks the constraint; the model fixes mistakes step-by-step
3. **Half-and-mirror with LLM polish** — Python guarantees the palindrome; the LLM makes it readable

**Prerequisites:**
```bash
# Install Ollama: https://ollama.com/download
ollama pull llama3.2       # 2 GB, fast on CPU
# Optionally: ollama pull mistral  or  ollama pull qwen2.5:7b
pip install ollama
```

A palindrome reads the same forwards and backwards (ignoring spaces and punctuation):
```
Was it a car or a cat I saw
▲                           ▲
wasitacaroracatisaw  ==  wasitacaroracatisaw (reversed)
```


## 0. Setup

In [ ]:
import re
import time
import ollama

# Check that the Ollama server is running and the model is available
MODEL = "llama3.2"   # change to "mistral", "qwen2.5:7b", etc. if you prefer

try:
    models = [m.model for m in ollama.list().models]
    if any(MODEL in m for m in models):
        print(f"Model '{MODEL}' is available.")
    else:
        print(f"Model '{MODEL}' not found. Available: {models}")
        print(f"Run: ollama pull {MODEL}")
except Exception as e:
    print(f"Could not connect to Ollama: {e}")
    print("Make sure Ollama is running: ollama serve")


## 1. Palindrome Utilities

In [ ]:
def clean_text(text: str) -> str:
    """Strip non-letters and lowercase."""
    return re.sub(r'[^a-z]', '', text.lower())

def is_palindrome(text: str) -> bool:
    c = clean_text(text)
    return bool(c) and c == c[::-1]

def palindrome_distance(text: str) -> int:
    """Number of mismatched character pairs. 0 = perfect palindrome."""
    c = clean_text(text)
    return sum(1 for i in range(len(c) // 2) if c[i] != c[-(i+1)])

def first_mismatch_hint(text: str) -> str:
    """Return a human-readable hint about the first mismatch."""
    c = clean_text(text)
    for i in range(len(c) // 2):
        if c[i] != c[-(i+1)]:
            return (
                f"Position {i} from the start is '{c[i]}' "
                f"but position {i} from the end is '{c[-(i+1)]}'. "
                f"Forward: '{c}' | Backward: '{c[::-1]}'"
            )
    return "No mismatch found."

# Sanity checks
for ex in [
    "A man a plan a canal Panama",
    "Never odd or even",
    "Step on no pets",
    "Not a palindrome at all",
]:
    ok = is_palindrome(ex)
    dist = palindrome_distance(ex)
    print(f"  {'OK' if ok else 'XX'}  [{dist:2d} err]  {ex}")


## 2. Approach 1: Direct Prompting

Ask the model directly for a palindrome.
We test several prompting styles to understand which gives the best hit rate.


In [ ]:
def ask_direct(style: str = "simple", model: str = MODEL) -> str:
    prompts = {
        "simple": (
            "Give me a palindrome sentence. "
            "It must read exactly the same forwards and backwards when you ignore spaces and punctuation. "
            "Reply with ONLY the palindrome, nothing else."
        ),
        "few_shot": (
            "Generate a new palindrome sentence — one that reads identically forwards and backwards "
            "(ignoring spaces and punctuation).\n\n"
            "Classic examples:\n"
            "- A man a plan a canal Panama\n"
            "- Never odd or even\n"
            "- Was it a car or a cat I saw\n"
            "- Step on no pets\n\n"
            "Your palindrome (reply with ONLY the sentence):"
        ),
        "chain_of_thought": (
            "I want a palindrome sentence. Think step by step:\n"
            "1. Choose a short word or name as the centre.\n"
            "2. Build outward symmetrically so that reading left-to-right and right-to-left gives the same letters.\n"
            "3. State only the final palindrome on the last line, prefixed with 'PALINDROME: '."
        ),
    }
    resp = ollama.chat(model=model, messages=[{"role": "user", "content": prompts[style]}])
    text = resp["message"]["content"].strip()
    # For chain-of-thought, extract the labelled line
    if style == "chain_of_thought" and "PALINDROME:" in text:
        text = text.split("PALINDROME:")[-1].strip().split("\n")[0].strip()
    return text

print("Testing direct prompting styles (5 attempts each):")
print()
for style in ["simple", "few_shot", "chain_of_thought"]:
    successes = 0
    print(f"Style: {style}")
    for attempt in range(5):
        result = ask_direct(style)
        ok = is_palindrome(result)
        dist = palindrome_distance(result)
        status = "PALINDROME" if ok else f"dist={dist}"
        print(f"  [{status:12s}] {result[:60]}")
        if ok:
            successes += 1
    print(f"  => {successes}/5 correct\n")


## 3. Approach 2: Iterative Refinement Loop

The model rarely produces a correct palindrome on the first try.
But it *can* fix a near-miss when shown exactly where it went wrong.

We run a Python loop:
1. Ask for a palindrome
2. Check it with Python
3. If wrong, report the exact mismatch and ask the model to fix it
4. Repeat up to `max_iter` times


In [ ]:
def refine_palindrome(model: str = MODEL, max_iter: int = 10, verbose: bool = True):
    """
    Iteratively ask the model to fix its palindrome attempts.
    Returns (final_text, iterations_taken, succeeded).
    """
    # Start with a few-shot prompt for the first attempt
    candidate = ask_direct("few_shot", model)

    for i in range(max_iter):
        ok = is_palindrome(candidate)
        dist = palindrome_distance(candidate)
        clean = clean_text(candidate)

        if verbose:
            status = "DONE" if ok else f"dist={dist}"
            print(f"  Iter {i+1:2d}: [{status:8s}] {candidate[:60]}")

        if ok:
            return candidate, i + 1, True

        hint = first_mismatch_hint(candidate)
        fix_prompt = (
            f'The sentence "{candidate}" is NOT a palindrome.\n'
            f"{hint}\n"
            f"Please correct it to be a true character palindrome "
            f"(same letters forwards and backwards, ignoring spaces/punctuation).\n"
            f"Reply with ONLY the corrected palindrome, nothing else."
        )
        resp = ollama.chat(model=model, messages=[{"role": "user", "content": fix_prompt}])
        candidate = resp["message"]["content"].strip().split("\n")[0].strip()

    return candidate, max_iter, is_palindrome(candidate)


print(f"Running iterative refinement with {MODEL}...")
print()
result, iters, success = refine_palindrome(verbose=True)
print()
print(f"Final: {result}")
print(f"Success: {success} after {iters} iterations")
print(f"Palindrome check: {is_palindrome(result)}")


In [ ]:
# Run 10 attempts and collect statistics
print(f"Running 10 independent attempts with iterative refinement ({MODEL})...")
print()

successes = []
failures = []

for run in range(10):
    result, iters, success = refine_palindrome(verbose=False)
    if success:
        successes.append((result, iters))
        print(f"  Run {run+1:2d}: SUCCESS in {iters} iter — {result}")
    else:
        failures.append(result)
        dist = palindrome_distance(result)
        print(f"  Run {run+1:2d}: failed  (dist={dist}) — {result[:50]}")

print()
print(f"Success rate: {len(successes)}/10")
if successes:
    avg_iters = sum(i for _, i in successes) / len(successes)
    print(f"Average iterations to success: {avg_iters:.1f}")
    print()
    print("Successful palindromes found:")
    for text, iters in successes:
        print(f"  ({iters} iter) {text}")


## 4. Approach 3: Half-and-Mirror with LLM Polish

A hybrid strategy that **guarantees** the palindrome property:

1. Ask the LLM to generate a short natural phrase (the "first half")
2. Python mirrors the characters deterministically — no LLM involved
3. Ask the LLM to add natural spacing/punctuation to the mirrored character sequence

The LLM contributes creativity (the seed phrase) and polish (the final presentation).
Python enforces correctness.


In [ ]:
def get_seed_phrase(model: str = MODEL) -> str:
    """Ask the model for a short natural phrase to use as the first half."""
    resp = ollama.chat(model=model, messages=[{
        "role": "user",
        "content": (
            "Give me a short English phrase of exactly 3-5 words "
            "(no punctuation, lowercase) suitable as the first half of a palindrome.\n"
            "Examples of good first halves:\n"
            "- step on no\n"
            "- was it a car\n"
            "- never odd or\n"
            "Reply with ONLY the phrase."
        )
    }])
    return resp["message"]["content"].strip().lower()

def mirror_characters(text: str) -> str:
    """Return text + reverse(text), creating an even-length character palindrome."""
    letters = clean_text(text)
    return letters + letters[::-1]

def polish_palindrome(raw_chars: str, model: str = MODEL) -> str:
    """Ask the model to add spaces/punctuation to a character palindrome."""
    resp = ollama.chat(model=model, messages=[{
        "role": "user",
        "content": (
            f"The following character sequence is a palindrome: '{raw_chars}'\n"
            "Re-read it and write it as a natural English sentence by adding spaces "
            "and minimal punctuation where they make sense. "
            "Do NOT change, add, or remove any letters — only add spaces and punctuation.\n"
            "Reply with ONLY the polished sentence."
        )
    }])
    return resp["message"]["content"].strip()

def half_mirror_pipeline(model: str = MODEL, n: int = 5):
    """Run n half-and-mirror attempts and report results."""
    print(f"Half-and-mirror pipeline ({n} attempts, {model}):")
    print()
    results = []
    for i in range(n):
        seed = get_seed_phrase(model)
        raw = mirror_characters(seed)
        polished = polish_palindrome(raw, model)
        ok = is_palindrome(raw)  # raw is always a palindrome; check polished too
        ok_polished = is_palindrome(polished)
        letters_match = clean_text(polished) == clean_text(raw)
        print(f"  Attempt {i+1}:")
        print(f"    Seed    : {seed}")
        print(f"    Raw     : {raw}  (palindrome: {ok})")
        print(f"    Polished: {polished}")
        print(f"    Polished palindrome: {ok_polished} | Letters preserved: {letters_match}")
        print()
        results.append({
            "seed": seed, "raw": raw, "polished": polished,
            "raw_ok": ok, "polished_ok": ok_polished, "letters_ok": letters_match
        })
    return results

results = half_mirror_pipeline(n=5)


In [ ]:
print("Summary:")
raw_ok = sum(1 for r in results if r['raw_ok'])
polished_ok = sum(1 for r in results if r['polished_ok'])
letters_ok = sum(1 for r in results if r['letters_ok'])
print(f"  Raw character palindromes (always correct): {raw_ok}/{len(results)}")
print(f"  Polished palindromes (letters unchanged):  {polished_ok}/{len(results)}")
print(f"  Letters preserved by polish step:          {letters_ok}/{len(results)}")
print()
print("Observation: the raw palindrome is ALWAYS correct (Python guaranteed it).")
print("The polish step may introduce letter changes — if so, validate and re-prompt.")


## 5. Bonus: Asking the Model to Judge Its Own Output

An Ollama model can also act as a judge. We ask it to rate a batch of palindrome candidates
on two axes: palindrome correctness and naturalness. This is useful when you have many candidates
and no external scorer.


In [ ]:
def llm_judge(candidates: list, model: str = MODEL) -> list:
    """Ask the model to evaluate a list of palindrome candidates."""
    numbered = "\n".join(f"{i+1}. {c}" for i, c in enumerate(candidates))
    prompt = (
        "Here are some sentences. For each:\n"
        "a) Is it a true palindrome? (yes/no)\n"
        "b) Does it read naturally as English? (1-5, where 5 is most natural)\n\n"
        f"{numbered}\n\n"
        "Respond in this format for each:\n"
        "1. palindrome=yes/no naturalness=N"
    )
    resp = ollama.chat(model=model, messages=[{"role": "user", "content": prompt}])
    return resp["message"]["content"]

# Generate some candidates to judge
candidates_to_judge = [
    "A man a plan a canal Panama",
    "Never odd or even",
    "Step on no pets",
    "Was it a car or a cat I saw",
    "I saw a dog and a cat today",   # non-palindrome
    "Racecar drivers race cars",     # non-palindrome
]

# Also cross-check with our Python validator
print("Python ground truth:")
for i, c in enumerate(candidates_to_judge, 1):
    ok = is_palindrome(c)
    print(f"  {i}. {'palindrome' if ok else 'NOT palindrome':14s} — {c}")

print()
print("LLM judge response:")
print(llm_judge(candidates_to_judge))
print()
print("Note: compare the LLM's palindrome assessment against the Python ground truth above.")


## 6. Comparison of Approaches

| Approach | Palindrome guaranteed? | Naturalness | Effort | Best for |
|---|---|---|---|---|
| Direct prompting | No (~10-25%) | High if correct | Minimal | Quick experiments |
| Iterative refinement | ~70-90% in 10 iter | Medium-High | Low | Practical generation |
| Half-and-mirror | Yes (raw) / maybe (polished) | Medium | Low | Guaranteed output |
| LLM as judge | N/A | Subjective | Minimal | Filtering candidates |

**Recommendation:**
- Start with **iterative refinement** — it requires no changes to the model and produces the most natural results
- Use **half-and-mirror** when you need a guaranteed palindrome (e.g. for a dataset or a game)
- Use **LLM-as-judge** to rank batches of candidates when you have many options

See also `palindrome_huggingface.ipynb` for constrained decoding and BERT bidirectional scoring.
